# notebooks/02_feature_engineering.ipynb

In [176]:
import sys, os

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath(os.path.join('..', 'scripts')))
from data_loader import create_panel_data

## Load Panel Data

In [177]:
panel_df = create_panel_data(frequency='5_min')

print('Data Head: ')
print(panel_df.head())
print('\nData Tail: ')
print(panel_df.tail())

/Users/fanzirui/Desktop/Sam_MacBook Pro/Duke/FA25/FINTECH 540/Project/scripts/data_loader.py:61: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  stacked = df.stack(dropna=False)
/Users/fanzirui/Desktop/Sam_MacBook Pro/Duke/FA25/FINTECH 540/Project/scripts/data_loader.py:61: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  stacked = df.stack(dropna=False)
/Users/fanzirui/Desktop/Sam_MacBook Pro/Duke/FA25/FINTECH 540/Project/scripts/data_loader.py:61: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the Wh

Data Head: 
                        rv       bpv      good       bad          rq
Date       Stock                                                    
2003-01-02 AAPL   6.493909  3.771960  5.102315  1.391595  152.729402
           AMGN   5.177506  3.635898  3.362921  1.814585   46.168213
           AMZN   9.886836  8.018926  6.074276  3.812559  200.751309
           AXP    4.448244  4.122573  3.337253  1.110991  114.695371
           BA     7.469396  7.016747  5.107386  2.362010  150.112379

Data Tail: 
                        rv       bpv      good       bad        rq
Date       Stock                                                  
2024-03-28 TRV    0.501900  0.450509  0.225921  0.275979  0.517683
           UNH    0.774552  0.762729  0.406755  0.367797  0.696658
           V      0.627872  0.445852  0.329977  0.297895  1.160977
           VZ     0.783853  0.706211  0.493097  0.290755  1.847938
           WMT    0.359440  0.336123  0.106481  0.252960  0.150975


/Users/fanzirui/Desktop/Sam_MacBook Pro/Duke/FA25/FINTECH 540/Project/scripts/data_loader.py:61: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  stacked = df.stack(dropna=False)


## Integrate VIX Data

In [178]:
START_DATE = '2003-01-02'
END_DATE = '2024-03-28'

vix_path = os.path.join('..', 'data', 'VIX_History.csv')
vix_data = pd.read_csv(vix_path)

vix_data['Date'] = pd.to_datetime(vix_data['DATE'], format='%m/%d/%Y')
vix_data = vix_data.set_index('Date')
vix = vix_data[['CLOSE']].rename(columns={'CLOSE': 'vix'})
vix = vix.loc[START_DATE:END_DATE]

panel_vix = panel_df.join(vix, on='Date')
print('Panel with VIX: ')
print(panel_vix.head())

Panel with VIX: 
                        rv       bpv      good       bad          rq    vix
Date       Stock                                                           
2003-01-02 AAPL   6.493909  3.771960  5.102315  1.391595  152.729402  25.39
           AMGN   5.177506  3.635898  3.362921  1.814585   46.168213  25.39
           AMZN   9.886836  8.018926  6.074276  3.812559  200.751309  25.39
           AXP    4.448244  4.122573  3.337253  1.110991  114.695371  25.39
           BA     7.469396  7.016747  5.107386  2.362010  150.112379  25.39


## Define Target Variable (Y_reg)

In [179]:
df = panel_vix.sort_index()

df['Y_reg'] = df.groupby('Stock')['rv'].shift(-1)

print('Example for AAPL: ')
display(df.loc[pd.IndexSlice[:, 'AAPL'], :].tail()[['rv', 'Y_reg']])

Example for AAPL: 


,,rv,Y_reg
Date,Stock,,
2024-03-22,AAPL,1.216548,0.682342
2024-03-25,AAPL,0.682342,0.425990
2024-03-26,AAPL,0.425990,0.959378
2024-03-27,AAPL,0.959378,0.563307
2024-03-28,AAPL,0.563307,NaN


## Engineer Features (X)

In [180]:
# Engineering HAR Features

df['rv_lag_1'] = df.groupby('Stock')['rv'].shift(1)
df['rv_rolling_5'] = df.groupby('Stock')['rv_lag_1'].rolling(window=5, min_periods=1).mean().reset_index(level=0, drop=True)
df['rv_rolling_22'] = df.groupby('Stock')['rv_lag_1'].rolling(window=22, min_periods=1).mean().reset_index(level=0, drop=True)

df['bpv_lag_1'] = df.groupby('Stock')['bpv'].shift(1)
df['bpv_rolling_5'] = df.groupby('Stock')['bpv_lag_1'].rolling(window=5, min_periods=1).mean().reset_index(level=0, drop=True)
df['good_lag_1'] = df.groupby('Stock')['good'].shift(1)
df['bad_lag_1'] = df.groupby('Stock')['bad'].shift(1)
df['bad_lag_5'] = df.groupby('Stock')['bad'].rolling(window=5, min_periods=1).mean().reset_index(level=0, drop=True)
df['rq_lag_1'] = df.groupby('Stock')['rq'].shift(1)
df['vix_lag_1'] = df.groupby('Stock')['vix'].shift(1)

epsilon = 1e-10
# df['bad_good_ratio_lag_1'] = df['bad_lag_1'] / (df['good_lag_1'] + epsilon)
# df['jump_ratio_lag_1'] = (df['rv_lag_1'] - df['bpv_lag_1']) / (df['rv_lag_1'] + epsilon)
# df['jump_ratio_lag_1'] = df['jump_ratio_lag_1'].clip(lower=0, upper=1)
df['rv_vix_interaction'] = df['rv_lag_1'] * df['vix_lag_1']
# df['rv_bpv_interaction'] = df['rv_lag_1'] * df['bpv_lag_1']
df['bad_rv_interaction'] = df['bad_lag_1'] * df['rv_lag_1']
# df['rv_rolling_std_1'] = df.groupby('Stock')['rv_lag_1'].shift(1).std()

display(df.tail())

rv       bpv      good       bad        rq    vix  \
Date       Stock                                                            
2024-03-28 TRV    0.501900  0.450509  0.225921  0.275979  0.517683  13.01   
           UNH    0.774552  0.762729  0.406755  0.367797  0.696658  13.01   
           V      0.627872  0.445852  0.329977  0.297895  1.160977  13.01   
           VZ     0.783853  0.706211  0.493097  0.290755  1.847938  13.01   
           WMT    0.359440  0.336123  0.106481  0.252960  0.150975  13.01   

                  Y_reg  rv_lag_1  rv_rolling_5  rv_rolling_22  bpv_lag_1  \
Date       Stock                                                            
2024-03-28 TRV      NaN  0.460120      0.475846       0.647585   0.493355   
           UNH      NaN  0.520918      0.501865       1.205146   0.422040   
           V        NaN  0.736163      0.904239       0.672596   0.575953   
           VZ       NaN  1.289133      0.798543       1.050153   0.821117   
           WMT      NaN  0.405749      0.493407       0.576572   0.478360   

                  bpv_rolling_5  good_lag_1  bad_lag_1  bad_lag_5   rq_lag_1  \
Date       Stock                                                               
2024-03-28 TRV         0.458957    0.323104   0.137016   0.232949   0.350452   
           UNH         0.448416    0.229649   0.291269   0.296011   0.603048   
           V           0.812983    0.290604   0.445559   0.656093   1.109700   
           VZ          0.696777    1.063019   0.226114   0.299748  13.769554   
           WMT         0.486568    0.193483   0.212266   0.250171   0.159196   

                  vix_lag_1  rv_vix_interaction  bad_rv_interaction  
Date       Stock                                                     
2024-03-28 TRV        12.78            5.880331            0.063044  
           UNH        12.78            6.657337            0.151727  
           V          12.78            9.408158            0.328004  
           VZ         12.78           16.475119            0.291491  
           WMT        12.78            5.185475            0.086127

In [181]:
df_model_ready = df.dropna()

print(f'Original panel size: {df.shape}')
print(f'Model-ready data size: {df_model_ready.shape}')

Original panel size: (160380, 19)
Model-ready data size: (153515, 19)


In [182]:
output_path = os.path.join('..', 'data', 'panel_data_model_ready.parquet')
df_model_ready.to_parquet(output_path)
display(df_model_ready.head())

rv       bpv      good       bad         rq    vix  \
Date       Stock                                                             
2003-01-03 AAPL   6.574494  5.893317  3.638035  2.936459  62.054316  24.68   
           AMGN   3.225009  3.043199  1.961131  1.263879  31.538602  24.68   
           AMZN   5.290480  4.212468  3.812314  1.478167  61.555140  24.68   
           AXP    3.394580  3.409980  1.725383  1.669197  13.330858  24.68   
           BA     3.064001  3.755029  1.403990  1.660010  13.424943  24.68   

                     Y_reg  rv_lag_1  rv_rolling_5  rv_rolling_22  bpv_lag_1  \
Date       Stock                                                               
2003-01-03 AAPL   5.992264  6.493909      6.493909       6.493909   3.771960   
           AMGN   2.460476  5.177506      5.177506       5.177506   3.635898   
           AMZN   5.818162  9.886836      9.886836       9.886836   8.018926   
           AXP    4.693210  4.448244      4.448244       4.448244   4.122573   
           BA     2.934207  7.469396      7.469396       7.469396   7.016747   

                  bpv_rolling_5  good_lag_1  bad_lag_1  bad_lag_5    rq_lag_1  \
Date       Stock                                                                
2003-01-03 AAPL        3.771960    5.102315   1.391595   2.164027  152.729402   
           AMGN        3.635898    3.362921   1.814585   1.539232   46.168213   
           AMZN        8.018926    6.074276   3.812559   2.645363  200.751309   
           AXP         4.122573    3.337253   1.110991   1.390094  114.695371   
           BA          7.016747    5.107386   2.362010   2.011010  150.112379   

                  vix_lag_1  rv_vix_interaction  bad_rv_interaction  
Date       Stock                                                     
2003-01-03 AAPL       25.39          164.880356            9.036889  
           AMGN       25.39          131.456870            9.395025  
           AMZN       25.39          251.026754           37.694144  
           AXP        25.39          112.940920            4.941961  
           BA         25.39          189.647958           17.642785